# GNN Encoder

Notebook-only version of Mohit's GNN stage for the Structural Coder project. Everything needed for this part is inside this notebook.

## Workflow

1. Pull the graph and node embeddings from Neo4j
2. Convert the graph into `torch_geometric.data.HeteroData`
3. Train a heterogeneous GNN with self-supervised link prediction
4. Evaluate the trained model
5. Export graph-aware embeddings for later Graph-RAG use

Core idea:

```text
original embedding -> GNN -> graph-aware embedding
```

In [ ]:
# Cell 1: Neo4j to PyG utilities

import argparse
import json
import os
from collections import defaultdict
from pathlib import Path
from typing import Any

import torch
from neo4j import GraphDatabase
from torch_geometric.data import HeteroData

DEFAULT_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
DEFAULT_USER = os.getenv("NEO4J_USER", "neo4j")
DEFAULT_PASSWORD = os.getenv("NEO4J_PASSWORD", "pytorch2026")


def normalize_label(labels: list[str] | None) -> str:
    if not labels:
        return "Unknown"
    return labels[0]


def fetch_nodes(session) -> list[dict[str, Any]]:
    query = """
    MATCH (n)
    WHERE n.embedding IS NOT NULL
    RETURN elementId(n) AS element_id,
           labels(n) AS labels,
           coalesce(n.qualified, n.name, n.title, n.url, elementId(n)) AS display_name,
           coalesce(n.docstring, n.description, n.text, n.note, '') AS summary,
           n.embedding AS embedding
    """
    records = []
    for record in session.run(query):
        embedding = record["embedding"]
        if not embedding:
            continue
        records.append(
            {
                "element_id": record["element_id"],
                "node_type": normalize_label(record["labels"]),
                "display_name": record["display_name"],
                "summary": record["summary"],
                "embedding": embedding,
            }
        )
    return records


def fetch_edges(session) -> list[dict[str, str]]:
    query = """
    MATCH (src)-[rel]->(dst)
    WHERE src.embedding IS NOT NULL AND dst.embedding IS NOT NULL
    RETURN elementId(src) AS src_id,
           type(rel) AS rel_type,
           elementId(dst) AS dst_id
    """
    return [
        {
            "src_id": record["src_id"],
            "relation": record["rel_type"],
            "dst_id": record["dst_id"],
        }
        for record in session.run(query)
    ]


def build_heterodata(
    nodes: list[dict[str, Any]],
    edges: list[dict[str, str]],
    add_reverse_edges: bool = True,
) -> tuple[HeteroData, dict[str, Any]]:
    data = HeteroData()
    grouped_nodes: dict[str, list[dict[str, Any]]] = defaultdict(list)
    id_to_local: dict[str, tuple[str, int]] = {}
    feature_dim: int | None = None

    for node in nodes:
        grouped_nodes[node["node_type"]].append(node)

    node_metadata: dict[str, list[dict[str, str]]] = {}
    for node_type in sorted(grouped_nodes):
        typed_nodes = grouped_nodes[node_type]
        embeddings = []
        metadata_rows = []
        for local_idx, node in enumerate(typed_nodes):
            vector = list(node["embedding"])
            if feature_dim is None:
                feature_dim = len(vector)
            elif len(vector) != feature_dim:
                raise ValueError(
                    f"Inconsistent embedding size for node {node['element_id']}: "
                    f"expected {feature_dim}, got {len(vector)}"
                )
            embeddings.append(vector)
            metadata_rows.append(
                {
                    "element_id": node["element_id"],
                    "display_name": node["display_name"],
                    "summary": node["summary"],
                }
            )
            id_to_local[node["element_id"]] = (node_type, local_idx)

        data[node_type].x = torch.tensor(embeddings, dtype=torch.float32)
        node_metadata[node_type] = metadata_rows

    edge_buckets: dict[tuple[str, str, str], set[tuple[int, int]]] = defaultdict(set)
    for edge in edges:
        src_info = id_to_local.get(edge["src_id"])
        dst_info = id_to_local.get(edge["dst_id"])
        if src_info is None or dst_info is None:
            continue

        src_type, src_idx = src_info
        dst_type, dst_idx = dst_info
        relation = edge["relation"]
        edge_buckets[(src_type, relation, dst_type)].add((src_idx, dst_idx))
        if add_reverse_edges:
            edge_buckets[(dst_type, f"rev_{relation}", src_type)].add((dst_idx, src_idx))

    for edge_type in sorted(edge_buckets):
        pairs = sorted(edge_buckets[edge_type])
        edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
        data[edge_type].edge_index = edge_index

    metadata = {
        "feature_dim": feature_dim or 0,
        "node_types": sorted(grouped_nodes),
        "edge_types": [list(edge_type) for edge_type in data.edge_types],
        "num_nodes_by_type": {
            node_type: int(data[node_type].num_nodes) for node_type in data.node_types
        },
        "num_edges_by_type": {
            "__".join(edge_type): int(data[edge_type].edge_index.size(1))
            for edge_type in data.edge_types
        },
        "node_metadata": node_metadata,
        "add_reverse_edges": add_reverse_edges,
    }
    return data, metadata


def save_graph_bundle(
    data: HeteroData,
    metadata: dict[str, Any],
    graph_path: Path,
    metadata_path: Path,
) -> None:
    graph_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(data, graph_path)
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")


def export_from_neo4j(
    uri: str,
    user: str,
    password: str,
    graph_path: Path,
    metadata_path: Path,
    add_reverse_edges: bool = True,
) -> tuple[HeteroData, dict[str, Any]]:
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session() as session:
            nodes = fetch_nodes(session)
            edges = fetch_edges(session)
        data, metadata = build_heterodata(nodes, edges, add_reverse_edges=add_reverse_edges)
        save_graph_bundle(data, metadata, graph_path, metadata_path)
        return data, metadata
    finally:
        driver.close()


def load_graph_bundle(graph_path: Path, metadata_path: Path) -> tuple[HeteroData, dict[str, Any]]:
    data = torch.load(graph_path)
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    return data, metadata


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Export the Neo4j KG into PyG HeteroData.")
    parser.add_argument("--uri", default=DEFAULT_URI, help="Neo4j Bolt URI")
    parser.add_argument("--user", default=DEFAULT_USER, help="Neo4j username")
    parser.add_argument("--password", default=DEFAULT_PASSWORD, help="Neo4j password")
    parser.add_argument(
        "--graph-path",
        default="outputs/hetero_graph.pt",
        help="Where to save the PyG graph bundle",
    )
    parser.add_argument(
        "--metadata-path",
        default="outputs/hetero_metadata.json",
        help="Where to save graph metadata",
    )
    parser.add_argument(
        "--no-reverse-edges",
        action="store_true",
        help="Skip adding synthetic reverse relations",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    graph_path = Path(args.graph_path)
    metadata_path = Path(args.metadata_path)
    data, metadata = export_from_neo4j(
        uri=args.uri,
        user=args.user,
        password=args.password,
        graph_path=graph_path,
        metadata_path=metadata_path,
        add_reverse_edges=not args.no_reverse_edges,
    )

    print(f"Saved graph to {graph_path.resolve()}")
    print(f"Saved metadata to {metadata_path.resolve()}")
    print(f"Node types: {len(data.node_types)}")
    print(f"Edge types: {len(data.edge_types)}")
    print(f"Feature dim: {metadata['feature_dim']}")


In [ ]:
# Cell 2: Heterogeneous GNN model

from typing import Iterable

import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv


def edge_type_to_key(edge_type: tuple[str, str, str]) -> str:
    return "__".join(edge_type)


class HeteroGraphEncoder(nn.Module):
    def __init__(
        self,
        metadata: tuple[list[str], list[tuple[str, str, str]]],
        hidden_channels: int = 256,
        out_channels: int = 128,
        num_layers: int = 2,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        if num_layers < 2:
            raise ValueError("num_layers must be at least 2")

        _, edge_types = metadata
        layer_dims = [hidden_channels] * (num_layers - 1) + [out_channels]
        self.convs = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        for out_dim in layer_dims:
            conv = HeteroConv(
                {
                    edge_type: SAGEConv((-1, -1), out_dim)
                    for edge_type in edge_types
                },
                aggr="sum",
            )
            self.convs.append(conv)

    def forward(
        self,
        x_dict: dict[str, torch.Tensor],
        edge_index_dict: dict[tuple[str, str, str], torch.Tensor],
    ) -> dict[str, torch.Tensor]:
        hidden = x_dict
        for layer_idx, conv in enumerate(self.convs):
            residual = hidden
            hidden = conv(hidden, edge_index_dict)
            for node_type, node_features in residual.items():
                if node_type not in hidden:
                    hidden[node_type] = node_features
            if layer_idx != len(self.convs) - 1:
                hidden = {
                    node_type: self.dropout(F.relu(node_features))
                    for node_type, node_features in hidden.items()
                }
        return {
            node_type: F.normalize(node_features, p=2, dim=-1)
            for node_type, node_features in hidden.items()
        }


class DistMultRelationDecoder(nn.Module):
    def __init__(
        self,
        edge_types: Iterable[tuple[str, str, str]],
        embedding_dim: int,
    ) -> None:
        super().__init__()
        self.relation_embeddings = nn.ParameterDict(
            {
                edge_type_to_key(edge_type): nn.Parameter(torch.empty(embedding_dim))
                for edge_type in edge_types
            }
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for parameter in self.relation_embeddings.values():
            nn.init.normal_(parameter, mean=1.0, std=0.1)

    def forward(
        self,
        z_dict: dict[str, torch.Tensor],
        edge_type: tuple[str, str, str],
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        src_type, _, dst_type = edge_type
        relation = self.relation_embeddings[edge_type_to_key(edge_type)]
        src_z = z_dict[src_type][edge_index[0]]
        dst_z = z_dict[dst_type][edge_index[1]]
        return (src_z * relation * dst_z).sum(dim=-1)


class HeteroLinkPredictor(nn.Module):
    def __init__(
        self,
        metadata: tuple[list[str], list[tuple[str, str, str]]],
        scored_edge_types: Iterable[tuple[str, str, str]],
        hidden_channels: int = 256,
        out_channels: int = 128,
        num_layers: int = 2,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.encoder = HeteroGraphEncoder(
            metadata=metadata,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            num_layers=num_layers,
            dropout=dropout,
        )
        self.decoder = DistMultRelationDecoder(
            edge_types=scored_edge_types,
            embedding_dim=out_channels,
        )

    def encode(self, data) -> dict[str, torch.Tensor]:
        return self.encoder(data.x_dict, data.edge_index_dict)

    def score_edges(
        self,
        z_dict: dict[str, torch.Tensor],
        edge_type: tuple[str, str, str],
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        return self.decoder(z_dict, edge_type, edge_index)


In [ ]:
# Cell 3: Evaluation helpers

import argparse
import json
from pathlib import Path
from typing import Any

import torch



def compute_classification_metrics(
    pos_scores: torch.Tensor,
    neg_scores: torch.Tensor,
) -> dict[str, float]:
    if pos_scores.numel() == 0 or neg_scores.numel() == 0:
        return {
            "roc_auc": float("nan"),
            "average_precision": float("nan"),
            "accuracy": float("nan"),
        }

    pos_scores = pos_scores.detach().cpu().float()
    neg_scores = neg_scores.detach().cpu().float()
    scores = torch.cat([pos_scores, neg_scores], dim=0)
    labels = torch.cat(
        [torch.ones_like(pos_scores), torch.zeros_like(neg_scores)],
        dim=0,
    )

    accuracy = ((scores >= 0).float() == labels).float().mean().item()

    num_pos = int(labels.sum().item())
    num_neg = int(labels.numel() - num_pos)
    order = torch.argsort(scores, stable=True)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(1, labels.numel() + 1, dtype=torch.float32)
    pos_rank_sum = ranks[labels == 1].sum().item()
    roc_auc = (pos_rank_sum - num_pos * (num_pos + 1) / 2.0) / (num_pos * num_neg)

    desc_order = torch.argsort(scores, descending=True, stable=True)
    sorted_labels = labels[desc_order]
    cumulative_hits = torch.cumsum(sorted_labels, dim=0)
    positions = torch.arange(1, labels.numel() + 1, dtype=torch.float32)
    precision_at_k = cumulative_hits / positions
    average_precision = (
        (precision_at_k * sorted_labels).sum().item() / max(num_pos, 1)
    )

    return {
        "roc_auc": float(roc_auc),
        "average_precision": float(average_precision),
        "accuracy": float(accuracy),
    }


@torch.no_grad()
def evaluate_split(
    model: HeteroLinkPredictor,
    message_graph,
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    split_name: str,
) -> dict[str, Any]:
    model.eval()
    z_dict = model.encode(message_graph)

    per_type: dict[str, dict[str, float]] = {}
    all_pos_scores = []
    all_neg_scores = []

    for edge_type, state in split_state.items():
        pos_edge_index = state.get(f"{split_name}_pos")
        neg_edge_index = state.get(f"{split_name}_neg")
        if pos_edge_index is None or neg_edge_index is None:
            continue
        if pos_edge_index.numel() == 0 or neg_edge_index.numel() == 0:
            continue

        pos_scores = model.score_edges(z_dict, edge_type, pos_edge_index)
        neg_scores = model.score_edges(z_dict, edge_type, neg_edge_index)
        per_type[edge_type_to_key(edge_type)] = compute_classification_metrics(
            pos_scores,
            neg_scores,
        )
        all_pos_scores.append(pos_scores)
        all_neg_scores.append(neg_scores)

    if not all_pos_scores:
        overall = {
            "roc_auc": float("nan"),
            "average_precision": float("nan"),
            "accuracy": float("nan"),
        }
    else:
        overall = compute_classification_metrics(
            torch.cat(all_pos_scores, dim=0),
            torch.cat(all_neg_scores, dim=0),
        )

    return {
        "split": split_name,
        "overall": overall,
        "per_type": per_type,
    }


def load_checkpoint_model(checkpoint_path: Path, message_graph) -> HeteroLinkPredictor:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    config = checkpoint["model_config"]
    model = HeteroLinkPredictor(
        metadata=message_graph.metadata(),
        scored_edge_types=config["scored_edge_types"],
        hidden_channels=config["hidden_channels"],
        out_channels=config["out_channels"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
    )
    model.load_state_dict(checkpoint["model_state"])
    return model


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Evaluate the trained hetero GNN.")
    parser.add_argument(
        "--message-graph-path",
        default="outputs/train_graph.pt",
        help="Message-passing graph used during evaluation",
    )
    parser.add_argument(
        "--checkpoint-path",
        default="outputs/best_model.pt",
        help="Saved model checkpoint",
    )
    parser.add_argument(
        "--split-state-path",
        default="outputs/split_state.pt",
        help="Saved split state from training",
    )
    parser.add_argument(
        "--split",
        default="test",
        choices=["val", "test"],
        help="Which split to evaluate",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    message_graph = torch.load(Path(args.message_graph_path), map_location="cpu")
    split_state = torch.load(Path(args.split_state_path), map_location="cpu")
    model = load_checkpoint_model(Path(args.checkpoint_path), message_graph)
    metrics = evaluate_split(model, message_graph, split_state, args.split)
    print(json.dumps(metrics, indent=2))


In [ ]:
# Cell 4: Training helpers

import argparse
import json
import os
import random
from pathlib import Path
from typing import Any

import torch
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling


DEFAULT_SUPERVISION_RELATIONS = {
    "IMPLEMENTS",
    "CONTAINS",
    "HAS_PARAM",
    "CALLS",
    "RELATED_TO",
    "REPLACES",
}


def empty_edge_index() -> torch.Tensor:
    return torch.empty((2, 0), dtype=torch.long)


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def reverse_edge_type(edge_type: tuple[str, str, str]) -> tuple[str, str, str]:
    src_type, relation, dst_type = edge_type
    return dst_type, f"rev_{relation}", src_type


def split_edge_index(
    edge_index: torch.Tensor,
    val_ratio: float,
    test_ratio: float,
    generator: torch.Generator,
) -> dict[str, torch.Tensor]:
    num_edges = edge_index.size(1)
    if num_edges == 0:
        return {"train": empty_edge_index(), "val": empty_edge_index(), "test": empty_edge_index()}

    perm = torch.randperm(num_edges, generator=generator)
    shuffled = edge_index[:, perm]

    if num_edges < 3:
        return {"train": shuffled, "val": empty_edge_index(), "test": empty_edge_index()}

    val_count = int(num_edges * val_ratio)
    test_count = int(num_edges * test_ratio)

    if val_ratio > 0 and val_count == 0:
        val_count = 1
    if test_ratio > 0 and test_count == 0 and num_edges - val_count > 1:
        test_count = 1

    while num_edges - val_count - test_count < 1:
        if test_count > 0:
            test_count -= 1
        elif val_count > 0:
            val_count -= 1
        else:
            break

    train_count = num_edges - val_count - test_count
    train_edges = shuffled[:, :train_count]
    val_edges = shuffled[:, train_count : train_count + val_count]
    test_edges = shuffled[:, train_count + val_count :]
    return {"train": train_edges, "val": val_edges, "test": test_edges}


def sample_negative_edges(
    edge_index: torch.Tensor,
    num_src_nodes: int,
    num_dst_nodes: int,
    num_samples: int,
) -> torch.Tensor:
    if num_samples <= 0:
        return empty_edge_index()
    return negative_sampling(
        edge_index=edge_index,
        num_nodes=(num_src_nodes, num_dst_nodes),
        num_neg_samples=num_samples,
        method="sparse",
    )


def select_supervised_edge_types(
    data,
    allowed_relations: set[str],
    min_edges: int,
) -> list[tuple[str, str, str]]:
    selected = []
    for edge_type in data.edge_types:
        _, relation, _ = edge_type
        if relation.startswith("rev_"):
            continue
        num_edges = int(data[edge_type].edge_index.size(1))
        if relation in allowed_relations and num_edges >= min_edges:
            selected.append(edge_type)
    return sorted(selected)


def build_split_state(
    data,
    supervised_edge_types: list[tuple[str, str, str]],
    val_ratio: float,
    test_ratio: float,
    seed: int,
) -> dict[tuple[str, str, str], dict[str, torch.Tensor]]:
    generator = torch.Generator().manual_seed(seed)
    split_state = {}

    for edge_type in supervised_edge_types:
        edge_index = data[edge_type].edge_index.cpu()
        src_type, _, dst_type = edge_type
        num_src_nodes = int(data[src_type].num_nodes)
        num_dst_nodes = int(data[dst_type].num_nodes)

        splits = split_edge_index(edge_index, val_ratio, test_ratio, generator)
        split_state[edge_type] = {
            "full_pos": edge_index,
            "train_pos": splits["train"],
            "val_pos": splits["val"],
            "test_pos": splits["test"],
            "val_neg": sample_negative_edges(
                edge_index,
                num_src_nodes,
                num_dst_nodes,
                int(splits["val"].size(1)),
            ),
            "test_neg": sample_negative_edges(
                edge_index,
                num_src_nodes,
                num_dst_nodes,
                int(splits["test"].size(1)),
            ),
        }
    return split_state


def build_train_graph(full_data, split_state) -> Any:
    train_data = full_data.clone()
    for edge_type, state in split_state.items():
        train_edges = state["train_pos"].clone()
        train_data[edge_type].edge_index = train_edges

        rev_edge = reverse_edge_type(edge_type)
        if rev_edge in train_data.edge_types:
            train_data[rev_edge].edge_index = train_edges.flip(0).contiguous()
    return train_data


@torch.no_grad()
def evaluate_model(
    model: HeteroLinkPredictor,
    message_graph,
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    split_name: str,
    device: torch.device,
) -> dict[str, Any]:
    model.eval()
    z_dict = model.encode(message_graph)

    per_type = {}
    all_pos_scores = []
    all_neg_scores = []

    for edge_type, state in split_state.items():
        pos_edge_index = state[f"{split_name}_pos"]
        neg_edge_index = state[f"{split_name}_neg"]
        if pos_edge_index.numel() == 0 or neg_edge_index.numel() == 0:
            continue

        pos_edge_index = pos_edge_index.to(device)
        neg_edge_index = neg_edge_index.to(device)
        pos_scores = model.score_edges(z_dict, edge_type, pos_edge_index)
        neg_scores = model.score_edges(z_dict, edge_type, neg_edge_index)

        per_type[edge_type_to_key(edge_type)] = compute_classification_metrics(
            pos_scores,
            neg_scores,
        )
        all_pos_scores.append(pos_scores.detach().cpu())
        all_neg_scores.append(neg_scores.detach().cpu())

    if all_pos_scores:
        overall = compute_classification_metrics(
            torch.cat(all_pos_scores, dim=0),
            torch.cat(all_neg_scores, dim=0),
        )
    else:
        overall = {
            "roc_auc": float("nan"),
            "average_precision": float("nan"),
            "accuracy": float("nan"),
        }

    return {"overall": overall, "per_type": per_type}


def train_one_epoch(
    model: HeteroLinkPredictor,
    optimizer: torch.optim.Optimizer,
    train_graph,
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    device: torch.device,
) -> tuple[float, dict[str, float]]:
    model.train()
    optimizer.zero_grad()
    z_dict = model.encode(train_graph)

    losses = []
    all_pos_scores = []
    all_neg_scores = []

    for edge_type, state in split_state.items():
        train_pos = state["train_pos"]
        if train_pos.numel() == 0:
            continue

        src_type, _, dst_type = edge_type
        neg_edge_index = sample_negative_edges(
            state["full_pos"],
            int(train_graph[src_type].num_nodes),
            int(train_graph[dst_type].num_nodes),
            int(train_pos.size(1)),
        )
        if neg_edge_index.numel() == 0:
            continue

        pos_edge_index = train_pos.to(device)
        neg_edge_index = neg_edge_index.to(device)

        pos_scores = model.score_edges(z_dict, edge_type, pos_edge_index)
        neg_scores = model.score_edges(z_dict, edge_type, neg_edge_index)
        logits = torch.cat([pos_scores, neg_scores], dim=0)
        labels = torch.cat(
            [torch.ones_like(pos_scores), torch.zeros_like(neg_scores)],
            dim=0,
        )
        losses.append(F.binary_cross_entropy_with_logits(logits, labels))
        all_pos_scores.append(pos_scores.detach().cpu())
        all_neg_scores.append(neg_scores.detach().cpu())

    if not losses:
        raise RuntimeError("No supervised edges were available for training.")

    loss = torch.stack(losses).mean()
    loss.backward()
    optimizer.step()

    train_metrics = compute_classification_metrics(
        torch.cat(all_pos_scores, dim=0),
        torch.cat(all_neg_scores, dim=0),
    )
    return float(loss.item()), train_metrics


def maybe_load_or_export_graph(
    graph_path: Path,
    metadata_path: Path,
    refresh_graph: bool,
    uri: str,
    user: str,
    password: str,
):
    if graph_path.exists() and metadata_path.exists() and not refresh_graph:
        return load_graph_bundle(graph_path, metadata_path)
    return export_from_neo4j(
        uri=uri,
        user=user,
        password=password,
        graph_path=graph_path,
        metadata_path=metadata_path,
        add_reverse_edges=True,
    )


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train a hetero GNN encoder on the Neo4j KG.")
    parser.add_argument("--uri", default=DEFAULT_URI, help="Neo4j Bolt URI")
    parser.add_argument("--user", default=DEFAULT_USER, help="Neo4j username")
    parser.add_argument("--password", default=DEFAULT_PASSWORD, help="Neo4j password")
    parser.add_argument("--graph-path", default="outputs/hetero_graph.pt", help="Saved/full graph path")
    parser.add_argument(
        "--metadata-path",
        default="outputs/hetero_metadata.json",
        help="Saved graph metadata path",
    )
    parser.add_argument(
        "--output-dir",
        default="outputs",
        help="Directory for checkpoints and split artifacts",
    )
    parser.add_argument(
        "--refresh-graph",
        action="store_true",
        help="Re-export the graph from Neo4j before training",
    )
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--hidden-channels", type=int, default=256)
    parser.add_argument("--out-channels", type=int, default=128)
    parser.add_argument("--num-layers", type=int, default=2)
    parser.add_argument("--dropout", type=float, default=0.2)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--val-ratio", type=float, default=0.1)
    parser.add_argument("--test-ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--min-supervision-edges",
        type=int,
        default=1,
        help="Skip relation types with fewer than this many edges",
    )
    parser.add_argument(
        "--supervision-relations",
        nargs="*",
        default=sorted(DEFAULT_SUPERVISION_RELATIONS),
        help="Relation names to use for link prediction supervision",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    set_seed(args.seed)

    graph_path = Path(args.graph_path)
    metadata_path = Path(args.metadata_path)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    full_data, _ = maybe_load_or_export_graph(
        graph_path=graph_path,
        metadata_path=metadata_path,
        refresh_graph=args.refresh_graph,
        uri=args.uri,
        user=args.user,
        password=args.password,
    )

    supervision_relations = set(args.supervision_relations)
    supervised_edge_types = select_supervised_edge_types(
        full_data,
        allowed_relations=supervision_relations,
        min_edges=args.min_supervision_edges,
    )
    if not supervised_edge_types:
        raise RuntimeError("No eligible supervision edges were found. Adjust the relation list or thresholds.")

    split_state = build_split_state(
        full_data,
        supervised_edge_types,
        val_ratio=args.val_ratio,
        test_ratio=args.test_ratio,
        seed=args.seed,
    )
    train_graph_cpu = build_train_graph(full_data, split_state)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_graph = train_graph_cpu.to(device)
    model = HeteroLinkPredictor(
        metadata=train_graph.metadata(),
        scored_edge_types=supervised_edge_types,
        hidden_channels=args.hidden_channels,
        out_channels=args.out_channels,
        num_layers=args.num_layers,
        dropout=args.dropout,
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    best_val_ap = float("-inf")
    best_checkpoint = None
    history = []

    for epoch in range(1, args.epochs + 1):
        train_loss, train_metrics = train_one_epoch(
            model,
            optimizer,
            train_graph,
            split_state,
            device,
        )
        val_metrics = evaluate_model(model, train_graph, split_state, "val", device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_metrics": train_metrics,
                "val_metrics": val_metrics,
            }
        )

        current_val_ap = val_metrics["overall"]["average_precision"]
        if current_val_ap == current_val_ap and current_val_ap > best_val_ap:
            best_val_ap = current_val_ap
            best_checkpoint = {
                "model_state": model.state_dict(),
                "model_config": {
                    "hidden_channels": args.hidden_channels,
                    "out_channels": args.out_channels,
                    "num_layers": args.num_layers,
                    "dropout": args.dropout,
                    "scored_edge_types": supervised_edge_types,
                },
                "best_val_metrics": val_metrics,
            }
            torch.save(best_checkpoint, output_dir / "best_model.pt")

        print(
            f"Epoch {epoch:03d} | loss={train_loss:.4f} "
            f"| train_ap={train_metrics['average_precision']:.4f} "
            f"| val_ap={val_metrics['overall']['average_precision']:.4f}"
        )

    if best_checkpoint is None:
        raise RuntimeError("Training completed but no valid checkpoint was produced.")

    torch.save(split_state, output_dir / "split_state.pt")
    torch.save(train_graph_cpu, output_dir / "train_graph.pt")

    final_model = HeteroLinkPredictor(
        metadata=train_graph.metadata(),
        scored_edge_types=supervised_edge_types,
        hidden_channels=args.hidden_channels,
        out_channels=args.out_channels,
        num_layers=args.num_layers,
        dropout=args.dropout,
    ).to(device)
    final_model.load_state_dict(best_checkpoint["model_state"])
    test_metrics = evaluate_model(final_model, train_graph, split_state, "test", device)

    summary = {
        "device": str(device),
        "graph_path": str(graph_path.resolve()),
        "metadata_path": str(metadata_path.resolve()),
        "supervised_edge_types": [list(edge_type) for edge_type in supervised_edge_types],
        "best_val_metrics": best_checkpoint["best_val_metrics"],
        "test_metrics": test_metrics,
        "history": history,
    }
    (output_dir / "training_summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    print("Saved artifacts:")
    print(f"- {output_dir / 'best_model.pt'}")
    print(f"- {output_dir / 'train_graph.pt'}")
    print(f"- {output_dir / 'split_state.pt'}")
    print(f"- {output_dir / 'training_summary.json'}")


In [ ]:
# Cell 5: Embedding export helpers

import argparse
import json
import re
from pathlib import Path

import torch
from neo4j import GraphDatabase



PROPERTY_NAME_PATTERN = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")


def validate_property_name(property_name: str) -> None:
    if not PROPERTY_NAME_PATTERN.fullmatch(property_name):
        raise ValueError(
            "Neo4j property names may only contain letters, numbers, and underscores, "
            "and must not start with a number."
        )


@torch.no_grad()
def build_export_rows(model: HeteroLinkPredictor, full_graph, metadata: dict) -> list[dict]:
    model.eval()
    z_dict = model.encode(full_graph)
    rows = []

    for node_type, embeddings in z_dict.items():
        typed_metadata = metadata["node_metadata"][node_type]
        if len(typed_metadata) != embeddings.size(0):
            raise ValueError(
                f"Metadata mismatch for node type {node_type}: "
                f"{len(typed_metadata)} rows vs {embeddings.size(0)} embeddings"
            )
        for idx, node_info in enumerate(typed_metadata):
            rows.append(
                {
                    "element_id": node_info["element_id"],
                    "node_type": node_type,
                    "display_name": node_info["display_name"],
                    "embedding": embeddings[idx].detach().cpu().tolist(),
                }
            )
    return rows


def write_embeddings_to_neo4j(
    rows: list[dict],
    uri: str,
    user: str,
    password: str,
    property_name: str,
    batch_size: int,
) -> None:
    validate_property_name(property_name)
    query = f"""
    UNWIND $rows AS row
    MATCH (n) WHERE elementId(n) = row.element_id
    SET n.{property_name} = row.embedding
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session() as session:
            for start in range(0, len(rows), batch_size):
                chunk = rows[start : start + batch_size]
                session.run(query, rows=chunk)
    finally:
        driver.close()


def load_model(checkpoint_path: Path, full_graph) -> HeteroLinkPredictor:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    config = checkpoint["model_config"]
    model = HeteroLinkPredictor(
        metadata=full_graph.metadata(),
        scored_edge_types=config["scored_edge_types"],
        hidden_channels=config["hidden_channels"],
        out_channels=config["out_channels"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
    )
    model.load_state_dict(checkpoint["model_state"])
    return model


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Export graph-aware node embeddings.")
    parser.add_argument("--graph-path", default="outputs/hetero_graph.pt", help="Full graph path")
    parser.add_argument(
        "--metadata-path",
        default="outputs/hetero_metadata.json",
        help="Graph metadata path",
    )
    parser.add_argument(
        "--checkpoint-path",
        default="outputs/best_model.pt",
        help="Trained model checkpoint",
    )
    parser.add_argument(
        "--output-jsonl",
        default="outputs/gnn_embeddings.jsonl",
        help="Where to save the exported embeddings as JSONL",
    )
    parser.add_argument(
        "--output-pt",
        default="outputs/gnn_embeddings.pt",
        help="Where to save the exported embeddings as a torch file",
    )
    parser.add_argument("--uri", default=DEFAULT_URI, help="Neo4j Bolt URI")
    parser.add_argument("--user", default=DEFAULT_USER, help="Neo4j username")
    parser.add_argument("--password", default=DEFAULT_PASSWORD, help="Neo4j password")
    parser.add_argument(
        "--write-to-neo4j",
        action="store_true",
        help="Write the exported embeddings back into Neo4j",
    )
    parser.add_argument(
        "--neo4j-property",
        default="gnn_embedding",
        help="Node property to use when writing back to Neo4j",
    )
    parser.add_argument("--write-batch-size", type=int, default=256)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    full_graph, metadata = load_graph_bundle(Path(args.graph_path), Path(args.metadata_path))
    model = load_model(Path(args.checkpoint_path), full_graph)
    rows = build_export_rows(model, full_graph, metadata)

    output_jsonl = Path(args.output_jsonl)
    output_pt = Path(args.output_pt)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    output_pt.parent.mkdir(parents=True, exist_ok=True)

    with output_jsonl.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")
    torch.save(rows, output_pt)

    if args.write_to_neo4j:
        write_embeddings_to_neo4j(
            rows=rows,
            uri=args.uri,
            user=args.user,
            password=args.password,
            property_name=args.neo4j_property,
            batch_size=args.write_batch_size,
        )

    print(f"Saved JSONL embeddings to {output_jsonl.resolve()}")
    print(f"Saved torch embeddings to {output_pt.resolve()}")
    if args.write_to_neo4j:
        print(f"Wrote embeddings back to Neo4j property '{args.neo4j_property}'")


## Configuration

In [ ]:
CONFIG = {
    'uri': DEFAULT_URI,
    'user': DEFAULT_USER,
    'password': DEFAULT_PASSWORD,
    'graph_path': Path('outputs/hetero_graph.pt'),
    'metadata_path': Path('outputs/hetero_metadata.json'),
    'output_dir': Path('outputs'),
    'epochs': 50,
    'hidden_channels': 256,
    'out_channels': 128,
    'num_layers': 2,
    'dropout': 0.2,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'val_ratio': 0.1,
    'test_ratio': 0.1,
    'seed': 42,
    'min_supervision_edges': 1,
    'supervision_relations': sorted(DEFAULT_SUPERVISION_RELATIONS),
    'refresh_graph': False,
}

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
set_seed(CONFIG['seed'])
CONFIG

## Step 1: Export Neo4j Graph To PyG

In [ ]:
full_data, metadata = maybe_load_or_export_graph(
    graph_path=CONFIG['graph_path'],
    metadata_path=CONFIG['metadata_path'],
    refresh_graph=CONFIG['refresh_graph'],
    uri=CONFIG['uri'],
    user=CONFIG['user'],
    password=CONFIG['password'],
)

print(full_data)
print(f'Feature dim: {metadata["feature_dim"]}')
print(f'Node types: {len(full_data.node_types)}')
print(f'Edge types: {len(full_data.edge_types)}')

In [ ]:
metadata['num_nodes_by_type'], metadata['num_edges_by_type']

## Step 2: Choose Supervision Relations And Build Splits

In [ ]:
supervised_edge_types = select_supervised_edge_types(
    full_data,
    allowed_relations=set(CONFIG['supervision_relations']),
    min_edges=CONFIG['min_supervision_edges'],
)

print('Supervised edge types:')
for edge_type in supervised_edge_types:
    print('  ', edge_type, '->', int(full_data[edge_type].edge_index.size(1)), 'edges')

split_state = build_split_state(
    full_data,
    supervised_edge_types=supervised_edge_types,
    val_ratio=CONFIG['val_ratio'],
    test_ratio=CONFIG['test_ratio'],
    seed=CONFIG['seed'],
)

train_graph_cpu = build_train_graph(full_data, split_state)
train_graph_cpu

## Step 3: Train The GNN

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_graph = train_graph_cpu.to(device)

model = HeteroLinkPredictor(
    metadata=train_graph.metadata(),
    scored_edge_types=supervised_edge_types,
    hidden_channels=CONFIG['hidden_channels'],
    out_channels=CONFIG['out_channels'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
)

history = []
best_val_ap = float('-inf')
best_checkpoint = None

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss, train_metrics = train_one_epoch(
        model=model,
        optimizer=optimizer,
        train_graph=train_graph,
        split_state=split_state,
        device=device,
    )
    val_metrics = evaluate_model(
        model=model,
        message_graph=train_graph,
        split_state=split_state,
        split_name='val',
        device=device,
    )

    epoch_record = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
    }
    history.append(epoch_record)

    current_val_ap = val_metrics['overall']['average_precision']
    if current_val_ap == current_val_ap and current_val_ap > best_val_ap:
        best_val_ap = current_val_ap
        best_checkpoint = {
            'model_state': model.state_dict(),
            'model_config': {
                'hidden_channels': CONFIG['hidden_channels'],
                'out_channels': CONFIG['out_channels'],
                'num_layers': CONFIG['num_layers'],
                'dropout': CONFIG['dropout'],
                'scored_edge_types': supervised_edge_types,
            },
            'best_val_metrics': val_metrics,
        }
        torch.save(best_checkpoint, CONFIG['output_dir'] / 'best_model.pt')

    print(
        f"Epoch {epoch:03d} | loss={train_loss:.4f} | train_ap={train_metrics['average_precision']:.4f} | val_ap={val_metrics['overall']['average_precision']:.4f}"
    )

if best_checkpoint is None:
    raise RuntimeError('Training finished without a valid checkpoint.')

torch.save(split_state, CONFIG['output_dir'] / 'split_state.pt')
torch.save(train_graph_cpu, CONFIG['output_dir'] / 'train_graph.pt')

print('Saved best checkpoint to', (CONFIG['output_dir'] / 'best_model.pt').resolve())

## Step 4: Evaluate Best Model On Test Edges

In [ ]:
best_model = HeteroLinkPredictor(
    metadata=train_graph.metadata(),
    scored_edge_types=supervised_edge_types,
    hidden_channels=CONFIG['hidden_channels'],
    out_channels=CONFIG['out_channels'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
).to(device)

best_model.load_state_dict(best_checkpoint['model_state'])
test_metrics = evaluate_model(
    model=best_model,
    message_graph=train_graph,
    split_state=split_state,
    split_name='test',
    device=device,
)

summary = {
    'device': str(device),
    'graph_path': str(CONFIG['graph_path'].resolve()),
    'metadata_path': str(CONFIG['metadata_path'].resolve()),
    'supervised_edge_types': [list(edge_type) for edge_type in supervised_edge_types],
    'best_val_metrics': best_checkpoint['best_val_metrics'],
    'test_metrics': test_metrics,
    'history': history,
}

(CONFIG['output_dir'] / 'training_summary.json').write_text(
    json.dumps(summary, indent=2),
    encoding='utf-8',
)

print(json.dumps(test_metrics, indent=2))

## Step 5: Export Final Graph-Aware Embeddings

In [ ]:
best_model_cpu = best_model.cpu()
rows = build_export_rows(best_model_cpu, full_data, metadata)

jsonl_path = CONFIG['output_dir'] / 'gnn_embeddings.jsonl'
pt_path = CONFIG['output_dir'] / 'gnn_embeddings.pt'

with jsonl_path.open('w', encoding='utf-8') as handle:
    for row in rows:
        handle.write(json.dumps(row) + '\n')

torch.save(rows, pt_path)

print('Saved JSONL embeddings to', jsonl_path.resolve())
print('Saved torch embeddings to', pt_path.resolve())
rows[0]

## Optional: Write Back To Neo4j

Uncomment the next cell only if you want the trained GNN vectors stored back in Neo4j as `gnn_embedding`.

In [ ]:
# write_embeddings_to_neo4j(
#     rows=rows,
#     uri=CONFIG['uri'],
#     user=CONFIG['user'],
#     password=CONFIG['password'],
#     property_name='gnn_embedding',
#     batch_size=256,
# )
# print('Wrote gnn_embedding back to Neo4j')